# YOLO26 + CBAM Experimental Matrix (E0 to E6)
### UAV Enemy Detection Research Suite

This notebook implements the complete research experimental matrix:
* **E0**: Baseline YOLO26 (No Attention)
* **E1**: 1× Backbone CBAM (Single Attention at P5)
* **E2**: 2× Backbone CBAM (Multiple Attention at P4 + P5)
* **E3**: 3× Backbone CBAM (Attention Depth at P3 + P4 + P5)
* **E4**: Neck CBAM (Location Comparison across 3× Neck Outputs)
* **E5**: Backbone + Neck Combined Attention
* **E6**: Best Model Analysis & Test Benchmark


In [ ]:
# 1. Environment & GPU Verification
import sys
from pathlib import Path
import torch
import ultralytics

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Ultralytics Version : {ultralytics.__version__}")
print(f"PyTorch Version     : {torch.__version__}")
print(f"CUDA Available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device          : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM          : {vram_gb:.2f} GB")
DEVICE = 0 if torch.cuda.is_available() else "cpu"


In [ ]:
# 2. Register CBAM Module Dynamically
from experiments.modules.cbam import register_cbam

register_cbam()
print("CBAM successfully registered into Ultralytics registry.")


In [ ]:
# 3. Dataset Configuration & Download
import os
from pathlib import Path

DATASET_DIR = PROJECT_ROOT / "datasets" / "enemy-detection-1-fps-v1"
DATASET_YAML = DATASET_DIR / "data_yolo26_local.yaml"
FALLBACK_YAML = DATASET_DIR / "data.yaml"

if not DATASET_YAML.is_file() and not FALLBACK_YAML.is_file():
    print("Dataset not found locally. Downloading from Roboflow...")
    api_key = os.environ.get("ROBOFLOW_API_KEY")
    if not api_key:
        from getpass import getpass
        api_key = getpass("Enter ROBOFLOW_API_KEY: ")
    from roboflow import Roboflow
    rf = Roboflow(api_key=api_key)
    project = rf.workspace("aaaa-g4pgj").project("enemy-detection-1-fps")
    dataset = project.version(1).download("yolo26")
    DATA_PATH = Path(dataset.location) / "data.yaml"
else:
    DATA_PATH = DATASET_YAML if DATASET_YAML.is_file() else FALLBACK_YAML

print(f"Using Dataset YAML: {DATA_PATH}")


In [ ]:
# 4. Shared Hyperparameters
# Model scale: 'n', 's', 'm', 'l', 'x'
MODEL_SIZE = "x"
EPOCHS = 100
BATCH_SIZE = 8 if MODEL_SIZE == "x" else 16
IMGSZ = 640
PATIENCE = 25
PRETRAINED = True  # Set to False to train from scratch

print(f"Training Config: Scale={MODEL_SIZE}, Epochs={EPOCHS}, Batch={BATCH_SIZE}, Imgsz={IMGSZ}")


## Experiment E0: Baseline YOLO26 (No Attention)


In [ ]:
# Train E0
from experiments.scripts.train import train_experiment

results_e0 = train_experiment(
    exp_id="E0",
    model_size=MODEL_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    data=str(DATA_PATH),
    pretrained=PRETRAINED,
    device=DEVICE,
    patience=PATIENCE
)


## Experiment E1: 1x Backbone CBAM (P5)


In [ ]:
# Train E1
from experiments.scripts.train import train_experiment

results_e1 = train_experiment(
    exp_id="E1",
    model_size=MODEL_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    data=str(DATA_PATH),
    pretrained=PRETRAINED,
    device=DEVICE,
    patience=PATIENCE
)


## Experiment E2: 2x Backbone CBAM (P4 + P5)


In [ ]:
# Train E2
from experiments.scripts.train import train_experiment

results_e2 = train_experiment(
    exp_id="E2",
    model_size=MODEL_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    data=str(DATA_PATH),
    pretrained=PRETRAINED,
    device=DEVICE,
    patience=PATIENCE
)


## Experiment E3: 3x Backbone CBAM (P3 + P4 + P5)


In [ ]:
# Train E3
from experiments.scripts.train import train_experiment

results_e3 = train_experiment(
    exp_id="E3",
    model_size=MODEL_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    data=str(DATA_PATH),
    pretrained=PRETRAINED,
    device=DEVICE,
    patience=PATIENCE
)


## Experiment E4: Neck CBAM (3x Neck Outputs)


In [ ]:
# Train E4
from experiments.scripts.train import train_experiment

results_e4 = train_experiment(
    exp_id="E4",
    model_size=MODEL_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    data=str(DATA_PATH),
    pretrained=PRETRAINED,
    device=DEVICE,
    patience=PATIENCE
)


## Experiment E5: Backbone + Neck Combined Attention


In [ ]:
# Train E5
from experiments.scripts.train import train_experiment

results_e5 = train_experiment(
    exp_id="E5",
    model_size=MODEL_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    data=str(DATA_PATH),
    pretrained=PRETRAINED,
    device=DEVICE,
    patience=PATIENCE
)


## Experiment E6: Final Model Analysis & Comparative Benchmark
Ranks models across E0 to E5, selects the best configuration, and runs full test evaluation.


In [ ]:
# Run E6 Automated Benchmark & Comparative Analysis
from experiments.scripts.analyze_e6 import run_e6_analysis

run_e6_analysis(
    project_dir=PROJECT_ROOT / "runs" / "experiments",
    primary_metric="val_map50",
    split="test",
    device=DEVICE,
    imgsz=IMGSZ
)


In [ ]:
# Display Comparison Table & Metric Charts
import pandas as pd
from IPython.display import display, Image

e6_dir = PROJECT_ROOT / "runs" / "experiments" / "E6_final_analysis"
comp_csv = e6_dir / "experiment_matrix_comparison.csv"
comp_plot = e6_dir / "matrix_comparison_metrics.png"

if comp_csv.is_file():
    df_results = pd.read_csv(comp_csv)
    print("=== FINAL EXPERIMENTAL MATRIX COMPARISON ===")
    display(df_results)

if comp_plot.is_file():
    display(Image(filename=str(comp_plot)))
